# Lección 07 - Patrón de Diseño de Planificación

Este cuaderno demuestra el **Patrón de Diseño de Planificación** para agentes de IA usando el Marco de Agentes de Microsoft.  
Aprenderás cómo descomponer solicitudes complejas de viaje en subtareas estructuradas, asignarlas a agentes especialistas,  
y ejecutar el plan resultante — todo usando una salida estructurada potenciada por modelos Pydantic.


## Configuración


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Descomposición de Tareas

La descomposición de tareas es el núcleo del patrón de diseño de planificación. En lugar de pedir a un solo agente que gestione una solicitud compleja de principio a fin, dividimos el problema en **subtareas** más pequeñas y bien definidas.  
Cada subtarea se asigna a un agente especialista (por ejemplo, vuelos, hoteles, actividades) con prioridades claras y un orden de dependencia.

Este enfoque ofrece varios beneficios:
- **Claridad**: cada subtarea tiene una sola responsabilidad.
- **Paralelismo**: las subtareas independientes pueden ejecutarse concurrentemente.
- **Confiabilidad**: las fallas están aisladas a subtareas individuales.
- **Seguimiento del presupuesto**: los costos se estiman por subtarea y se consolidan.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Creación de un Agente de Planificación con Salida Estructurada

El agente de planificación actúa como un **coordinador de recepción**. Dada una solicitud de viaje de alto nivel, produce un `TravelPlan` estructurado, descomponiendo la solicitud en subtareas, estableciendo prioridades e identificando dependencias para que un conserje o una capa de ejecución puedan llevar a cabo el trabajo.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Ejecutar un Plan con Herramientas Especializadas

Una vez que el agente de recepción ha elaborado un plan estructurado, el **agente conserje** lo ejecuta.
Cada herramienta especializada maneja una categoría de subtarea (vuelos, hoteles, actividades). El conserje
itera a través de las subtareas del plan en orden de dependencia y envía cada una a la
herramienta correspondiente.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Resumen

En esta lección aprendiste el **Patrón de Diseño de Planificación** para agentes de IA:

1. **Descomposición de Tareas** — Un agente planificador de recepción divide una solicitud de viaje compleja en subtareas estructuradas usando modelos Pydantic, asignando cada una a un agente especialista con prioridades y dependencias.
2. **Salida Estructurada** — Al pasar un `response_format`, el agente devuelve un objeto validado `TravelPlan` en lugar de texto libre, lo que hace que el procesamiento posterior sea confiable.
3. **Ejecución del Plan** — Un agente conserje recorre las subtareas usando herramientas especializadas (`book_flight`, `reserve_hotel`, `book_activity`) para ejecutar el plan y reportar resultados.

Este patrón separa *qué hacer* (planificación) de *cómo hacerlo* (ejecución), haciendo que los agentes sean más modulares, testeables y fáciles de extender.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por lograr la mayor precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda la traducción profesional realizada por humanos. No nos hacemos responsables de ningún malentendido o interpretación errónea derivada del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
